# 01 - Quickstart: program usages

Load the measured program-usage matrix for one context, annotate the 32
programs with their top genes from the pinned shared basis, and find the
top compounds of one program. Terms: a **program** is one of 32
coordinated gene-expression patterns shared across all contexts; a
compound's **usage** of a program is how strongly that program is
expressed in its measured RNA response. Everything here is measured
data from `core/`. Runs with numpy + pandas + pyarrow only.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the package root (works whether the notebook runs from examples/
# or from the package root).
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "core" / "basis" / "basis_registry.json").exists())
sys.path.insert(0, str(ROOT / "src"))
print("package root located (all paths below are relative to it)")


package root located (all paths below are relative to it)


## Load usages, basis, and the gene panel

The usage matrix is `(n_compounds, 32)`; row *i* is the compound named in
row *i* of the compounds parquet (the row-alignment contract). Column *j*
is program P*j*+1 of the pinned basis `shared_program_basis_v1`. We use
`zel039_aec7`, the context of the flagship heat-shock anchor lead.

In [2]:
from zscreen_program_package import data

CTX = "zel039_aec7"
usages, compounds = data.load_usages(CTX, root=ROOT)
basis = data.load_basis(k=32, root=ROOT)
panel = data.panel_genes(root=ROOT)
print("usages:", usages.shape, "| compounds:", len(compounds),
      "| basis:", basis.shape, "| panel:", len(panel))


usages: (20813, 32) | compounds: 20813 | basis: (32, 6000) | panel: 6000


## Annotate the programs with their top genes

Each basis row is one program's non-negative gene loadings over the
6,000-gene harmonized panel. Gene symbols come only from the panel file
(never from any context gene panel). Listing the top-loading genes per
program makes the programs readable: you can find the heat-shock program
by its HSP genes, with no prior annotation.

In [3]:
TOP_N = 12
CANONICAL_HSR = ("HSPA1A", "HSPA1B", "HSPA6", "DNAJB1", "HSPH1", "HSPD1",
                 "HSP90AA1", "HSPB1")
top_genes = {
    f"P{j + 1:02d}": panel.iloc[np.argsort(-basis[j])[:TOP_N]]["gene"].tolist()
    for j in range(basis.shape[0])
}
top30 = {
    f"P{j + 1:02d}": panel.iloc[np.argsort(-basis[j])[:30]]["gene"].tolist()
    for j in range(basis.shape[0])
}
annotation = pd.DataFrame({
    "program": list(top_genes),
    "top_genes": [", ".join(g) for g in top_genes.values()],
    "canonical_hsr_in_top30": [
        [g for g in CANONICAL_HSR if g in top30[p]] for p in top_genes],
})
annotation["n_canonical"] = annotation["canonical_hsr_in_top30"].map(len)
hsr = annotation.sort_values(["n_canonical", "program"], ascending=[False, True]).head(5)
print("programs richest in canonical heat-shock genes (HSPA1A/DNAJB1/HSPH1/...):")
print(hsr.assign(canonical_hsr_in_top30=hsr["canonical_hsr_in_top30"].map(", ".join))
      .to_string(index=False))


programs richest in canonical heat-shock genes (HSPA1A/DNAJB1/HSPH1/...):
program                                                                                    top_genes                        canonical_hsr_in_top30  n_canonical
    P20     UBC, HSP90AA1, UBB, HSPD1, LINC00486, SAT1, MALAT1, HSPA8, HSP90AB1, CD63, HSP90B1, RPS2 HSPA1A, DNAJB1, HSPH1, HSPD1, HSP90AA1, HSPB1            6
    P16            SAT1, HSP90AA1, HSPD1, HSPA8, HSPE1, UBB, RPS28, RPL39, BTF3, MALAT1, RPL18, NACA        HSPA1B, DNAJB1, HSPD1, HSP90AA1, HSPB1            5
    P28 MT-CO1, HSP90AA1, LINC00486, HSPA8, HSP90AB1, UBC, MT-CO2, HSPD1, MT-ND4, MT-CO3, UBB, ACTG1                 HSPH1, HSPD1, HSP90AA1, HSPB1            4
    P08             COX4I1, GSTP1, MYL6, SNHG29, UBB, RPL18, RPL36AL, TXN, RPS27L, FAU, ZFAS1, RPS28                               HSPD1, HSP90AA1            2
    P10     RPS27L, COX4I1, UBC, MYL6, TMEM258, CD63, RPL36AL, HSP90B1, HINT1, MALAT1, SNHG29, HSPA5                          

## Find a program's top compounds

Take the heat-shock program identified above and rank compounds by their
usage of it,
then join the building-block recipes. The anchor lead ZSH-0001
(`annex_hypotheses/anchor_leads.csv`) says carriers of bb0 level
`BB_2085420374` drive the HSF1/heat-shock program in this context, so
carriers of that level should pile up at the top of the ranking. This is
a measured-data cross-check of that lead's program assignment, computed
from `core/` alone.

In [4]:
program = hsr.iloc[0]["program"]
j = int(program[1:]) - 1
ranked = compounds.assign(usage=usages[:, j]).sort_values("usage", ascending=False)
recipes = data.load_recipes(root=ROOT)
top = ranked.head(50).merge(recipes, on="public_compound_id", how="left")

LEVEL = "BB_2085420374"
carriers_top = int((top["bb0"] == LEVEL).sum())
background = float((recipes["bb0"] == LEVEL).mean())
print(f"program {program} (top genes: {', '.join(top_genes[program][:5])}, ...)")
print(f"bb0={LEVEL} carriers in the top 50 compounds by usage: {carriers_top}/50")
print(f"background carrier rate across all recipes: {background:.1%}")
print()
print(top[["public_compound_id", "usage", "bb0", "bb1", "bb2"]].head(10).to_string(index=False))


program P20 (top genes: UBC, HSP90AA1, UBB, HSPD1, LINC00486, ...)
bb0=BB_2085420374 carriers in the top 50 compounds by usage: 29/50
background carrier rate across all recipes: 1.4%

public_compound_id     usage           bb0           bb1           bb2
  CPD_915793104368 36.946960 BB_2085420374 BB_0549297364 BB_6014564733
  CPD_858382165271 33.153976 BB_2085420374 BB_0549297364 BB_3217225768
  CPD_676680382846 32.481483 BB_2085420374 BB_6821040250 BB_8384776618
  CPD_323869603457 32.040745 BB_2085420374 BB_4318016053 BB_5134356880
  CPD_707111120228 31.474669 BB_8506073726 BB_2585216724 BB_8489537894
  CPD_259759092666 31.085455 BB_8506073726 BB_8687334682 BB_0001826021
  CPD_090819724731 30.964333 BB_8506073726 BB_5372566179 BB_3354738261
  CPD_828483389703 30.615681 BB_6699958170 BB_0094546068 BB_1564494047
  CPD_895603562885 29.513947 BB_0995925376 BB_5853818500 BB_8384776618
  CPD_606868200088 29.401077 BB_2085420374 BB_4825242175 BB_8523974361


The ranking recovers the anchor story from measured data alone: the
program whose top genes are the canonical heat-shock genes is used most
strongly by carriers of the anchor building block. For the full evidence
chain (control measurement, neighborhood enrichment, nulls, kill/confirm
experiment), see `annex_hypotheses/README.md` and lead ZSH-0001.